# I-3D — Full Image-to-3D-to-Motion Pipeline + Interactive Colab 3D Studio

**Source:** [Apache0ne/I-3D](https://github.com/Apache0ne/I-3D) • **Research workflow** • **One consistent rigged 3D person, two actual 3D poses, one generated animation**.

This notebook orchestrates the **original I-3D scripts** and the six vendored upstream implementations. It uses separate Python environments because the Hi3D, Hunyuan, TokenRig and MotionBricks CUDA/PyTorch stacks differ. It does not reimplement or silently replace the pretrained models.

**Runtime:** NVIDIA GPU, substantial system RAM and free disk; A100/H100-class GPU recommended for attempting the heavier geometry stage. Hi3D 1536²/³-level sparse resolution is not guaranteed on smaller GPUs. Use 512 as the initial geometry resolution. The GPU dependencies may require environment-specific adjustments.

Run cells in order. Heavy model downloads are intentional and licensing terms remain in force. This notebook has **not been executed or validated** in this conversation.


## Pipeline

1. Clone existing `main` from GitHub and select Colab working directories.
2. Install required, **isolated** model environments and download external model assets.
3. Upload **one** picture of a person.
4. Generate coarse Hunyuan3D-2mini geometry, refine with Hi3D, and paint PBR textures with Hunyuan3D-Paint 2.1.
5. Generate the **single** character skeleton and skinning with TokenRig.
6. Convert it to the actual SOMA 30-joint reference rig through I-3D's axis-aware Blender bridge.
7. Pose the **same 3D character** in Blender at frame 1 and frame 2, either using the notebook pose controls below or a locally edited `.blend`.
8. Use MotionBricks to generate motion, apply it to the same mesh and export animated GLB.

**Coordinate conventions:** NVIDIA SOMA right-handed Y-up → Blender right-handed Z-up via +90° X rotation. You must specify which direction the generated person visually faces. Only positive uniform scale is permitted. CPU math checks are included in the GitHub repository but are *not run in this notebook by default*.

**License notice:** Hi3D and Tencent Hunyuan models use their own licenses (including possible branding, distribution or geographic conditions). MotionBricks model weights and sample clips have separate terms from its Apache-2.0 code. Credits alone do not grant rights. Consult the [full license list](https://github.com/Apache0ne/I-3D#licenses-and-citations).


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, textwrap

ROOT = Path("/content/I-3D")
WORK = ROOT / "outputs" / "character"
ENVS = Path("/content/i3d_envs")
GEOMETRY_RESOLUTION = 512      # choices: 512, 1024, 1536
HI3D_STEPS = 25
CHARACTER_FORWARD = "+X"       # Blender world forward: +X, -X, +Y, -Y
MOTION_PRECISION = "fp16"       # fp16 or fp32
BLENDER = shutil.which("blender") or "/usr/bin/blender"

def run(*args, cwd=None, env=None):
    cmd = [str(a) for a in args]
    print("RUN:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=str(cwd or ROOT), env=env, check=True)

def required(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing output artifact: {path}")
    return path

print("Configured", dict(root=str(ROOT), output=str(WORK), forward=CHARACTER_FORWARD, geometry=GEOMETRY_RESOLUTION, precision=MOTION_PRECISION))


## 1. Clone current I-3D main and prepare Colab

The upstream **source code** is already vendored in the main repository. Pretrained model checkpoints, CUDA extensions and Git LFS assets are handled separately below.


In [ ]:
os.chdir("/content")
if not ROOT.exists():
    run("git", "clone", "--branch", "main", "--single-branch",
        "https://github.com/Apache0ne/I-3D.git", str(ROOT), cwd="/content")
else:
    print("Using existing checkout:", ROOT)
run("git", "rev-parse", "--short", "HEAD")
WORK.mkdir(parents=True, exist_ok=True)
print("Repository ready:", ROOT)


In [ ]:
run("bash", "-lc",
    "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git-lfs blender libgl1 libglib2.0-0")
run("git", "lfs", "install")
BLENDER = shutil.which("blender") or "/usr/bin/blender"
run("bash", "-lc",
    "command -v uv >/dev/null || python -m pip install -q uv")
UV = shutil.which("uv") or "/usr/local/bin/uv"
run(UV, "python", "install", "3.10", "3.11", "3.12")
for ver, label in (("3.10","hi3d"),("3.10","paint"),("3.11","tokenrig"),("3.12","motion")):
    folder = ENVS / label
    if not folder.exists():
        run(UV, "venv", "--seed", "--python", ver, str(folder))
HI_PY = str(ENVS/"hi3d"/"bin"/"python")
PAINT_PY = str(ENVS/"paint"/"bin"/"python")
TOKEN_PY = str(ENVS/"tokenrig"/"bin"/"python")
MOTION_PY = str(ENVS/"motion"/"bin"/"python")
print("Separate runtimes:", HI_PY, PAINT_PY, TOKEN_PY, MOTION_PY, sep="\n")


## 2. Install Hi3D + Hunyuan3D-2mini source environment

Hi3D's verified Linux stack specifies Python 3.10, CUDA 12.4, Torch 2.5.0, and source-built attention/rasterization dependencies. This installation follows the upstream approach; it may require adapting `nvcc` or CUDA build settings on a particular Colab image. Installation failure is reported rather than replaced by a mock model.


In [ ]:
HI = ROOT / "vendor" / "Hi3Dv1.0"
HUNY = ROOT / "vendor" / "Hunyuan3D-2"
run(UV, "pip", "install", "--python", HI_PY,
    "torch==2.5.0", "torchvision==0.20.0", "--index-url", "https://download.pytorch.org/whl/cu124")
run(UV, "pip", "install", "--python", HI_PY, "packaging", "ninja", "psutil", "wheel", "setuptools")
run(UV, "pip", "install", "--python", HI_PY, "--no-build-isolation", "flash-attn==2.7.4.post1")
run(UV, "pip", "install", "--python", HI_PY, "-c", str(HI/"docker"/"constraints.txt"),
    "-e", str(HI)+"[inference,demo]")
run(UV, "pip", "install", "--python", HI_PY, "-r", str(HUNY/"requirements.txt"))
run(UV, "pip", "install", "--python", HI_PY, "-e", str(HUNY))
print("Hi3D and Hunyuan3D-2mini source dependencies installed.")


## 3. Install Hunyuan3D-Paint v2.1 in a separate Python 3.10 environment

The paint pipeline includes a CUDA rasterizer and uses a separate PBR texture model. Its dependencies must not overwrite Hi3D's pinned packages.


In [ ]:
PAINT = ROOT / "vendor" / "Hunyuan3D-2.1"
run(UV, "pip", "install", "--python", PAINT_PY,
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124")
run(UV, "pip", "install", "--python", PAINT_PY, "-r", str(PAINT/"requirements.txt"))
run(UV, "pip", "install", "--python", PAINT_PY, "-e", str(PAINT/"hy3dpaint"/"custom_rasterizer"))
print("Hunyuan Paint environment installed.")


In [ ]:
# Hunyuan Paint's external RealESRGAN restoration weights.
from urllib.request import urlretrieve
ESRGAN = PAINT / "hy3dpaint" / "ckpt" / "RealESRGAN_x4plus.pth"
ESRGAN.parent.mkdir(parents=True, exist_ok=True)
if not ESRGAN.exists():
    urlretrieve("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth", str(ESRGAN))
print("RealESRGAN:", ESRGAN)


## 4. Install TokenRig / SkinTokens

Uses the original TokenRig inference code and its associated SkinTokens weight decoder; the source model downloads its checkpoints via its own `download.py`.


In [ ]:
TOKEN = ROOT / "vendor" / "SkinTokens"
run(UV, "pip", "install", "--python", TOKEN_PY,
    "torch==2.7.0", "torchvision==0.22.0", "torchaudio==2.7.0",
    "--index-url", "https://download.pytorch.org/whl/cu128")
run(UV, "pip", "install", "--python", TOKEN_PY, "-r", str(TOKEN/"requirements.txt"))
run(UV, "pip", "install", "--python", TOKEN_PY, "--no-build-isolation", "flash-attn")
run(TOKEN_PY, "download.py", "--model", cwd=TOKEN)
print("TokenRig source and checkpoints prepared.")


## 5. MotionBricks runtime and original NVIDIA assets

The MotionBricks runtime requires Python >=3.12 and TensorRT. Its `git lfs` assets are not part of the I-3D source commit. The hydrator downloads the genuine SOMA USD reference and selected FP32/mixed-FP16 ONNX graph.


In [ ]:
MOTION = ROOT / "vendor" / "motionbricks"
run(UV, "pip", "install", "--python", MOTION_PY, "-e", str(MOTION))
run(sys.executable, str(ROOT/"scripts"/"hydrate_motionbricks.py"), "--precision", MOTION_PRECISION)
SOMA_USD = MOTION / "motionbricks" / "assets" / "human" / "soma_base_skel_minimal_no_root.usd"
required(SOMA_USD)
print("Actual SOMA USD:", SOMA_USD, SOMA_USD.stat().st_size, "bytes")


## 6. Upload one reference image

Prefer a full-body person, visible limbs, and an upright A-pose or T-pose. The image is the only source image; later poses are edited in **3D**, not regenerated from images.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one PNG/JPEG image")
name, data = next(iter(uploaded.items()))
IMAGE = WORK / Path(name).name
IMAGE.write_bytes(data)
print("Input:", IMAGE)


## Live I-3D 3D Studio — runs entirely inside Colab

The viewer opens in the notebook, continuously discovers completed model files, displays PBR materials, compares pipeline stages, inspects joint weights and plays the generated animation. It also provides **START/TARGET SOMA 3D pose editing** without opening Blender. The browser handles rendering; the model GPU stays dedicated to inference.

The workspace requires internet access to load the Three.js browser libraries. Run the viewer again at the later stage to bring it next to the finished outputs.


In [ ]:
# Interactive Colab 3D studio: starts a local viewer server, no external web app.
from google.colab import output
import sys, socket
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from scripts.viewer_server import launch as launch_i3d_viewer
PORT = 8765
if "_i3d_viewer" not in globals():
    _i3d_viewer = launch_i3d_viewer(WORK, port=PORT)
    print("I-3D viewer server started", PORT)
else:
    print("Reusing existing I-3D viewer server", PORT)
output.serve_kernel_port_as_iframe(PORT, height=950)


## 7. Coarse 3D mesh + Hi3D refinement

Runs the original Hi3D `demo/image_to_mesh.py`, including its Hunyuan3D-2mini adapter and background removal. Hi3D 512 is the conservative starting configuration. Generated mesh does not include PBR yet.


In [ ]:
GEOMETRY = WORK / "01_geometry.glb"
if not GEOMETRY.exists():
    run(HI_PY, "demo/image_to_mesh.py",
        "--image", IMAGE, "--output", GEOMETRY,
        "--resolution", GEOMETRY_RESOLUTION,
        "--num-inference-steps", HI3D_STEPS,
        "--remove-background", cwd=HI)
required(GEOMETRY)
print("Hi3D refined geometry:", GEOMETRY)


## 8. UV and PBR texture generation

This uses the Hunyuan3D-Paint v2.1 *model inference*, not a procedural material placeholder. The mesh and image are fed to the original paint integration script.


In [ ]:
TEXTURED = WORK / "02_textured.glb"
if not TEXTURED.exists():
    run(PAINT_PY, ROOT/"scripts"/"paint_hunyuan.py",
        "--mesh", GEOMETRY, "--image", IMAGE, "--output", TEXTURED,
        "--resolution", 512, "--views", 6, cwd=PAINT)
required(TEXTURED)
print("PBR textured character:", TEXTURED)


## 9. Automatic skeleton + skinning with TokenRig

TokenRig predicts its own skeleton and skinning weights and `--use_transfer` preserves the original character's texture and scale. I-3D then converts that output onto the **canonical SOMA** rig.


In [ ]:
TOKENRIG_GLB = WORK / "03_tokenrig.glb"
if not TOKENRIG_GLB.exists():
    run(TOKEN_PY, "demo.py", "--input", TEXTURED,
        "--output", TOKENRIG_GLB, "--use_transfer", cwd=TOKEN)
required(TOKENRIG_GLB)
print("TokenRig character:", TOKENRIG_GLB)


## 10. Extract the real SOMA skeleton and build the I-3D skinning bridge

This imports the canonical **30-joint** skeleton from NVIDIA's USD reference, maps TokenRig's weights onto it, and exports an animated-rig-compatible GLB. The mapped weights are approximate and should be inspected for deforming limbs before production use.

**Direction matters:** `CHARACTER_FORWARD` must match the person's *visual* forward direction in Blender. No mirror is applied. The script rejects inverted anatomical labels and implausible rig alignment.


In [ ]:
REFERENCE = WORK / "soma_reference.json"
SOMA_GLB = WORK / "04_soma_character.glb"
BRIDGE = SOMA_GLB.with_suffix(".bridge.json")
if not REFERENCE.exists():
    run(MOTION_PY, ROOT/"scripts"/"export_soma_reference.py",
        "--usd", SOMA_USD, "--output", REFERENCE)
if not SOMA_GLB.exists():
    run(BLENDER, "-b", "--python", ROOT/"scripts"/"blender_bridge.py", "--",
        "--input", TOKENRIG_GLB, "--reference", REFERENCE,
        "--output", SOMA_GLB, "--forward", CHARACTER_FORWARD)
required(SOMA_GLB)
required(BRIDGE)
print(json.dumps(json.loads(BRIDGE.read_text()), indent=2))


## In-notebook 3D poser — use the real character, not another image

**Primary workflow:** open the live 3D Studio below; in the **Primary 3D stage** select **SOMA 30-joint character**. Select and rotate bones using the gizmo or numeric controls; move the root via X/Y/Z fields. Save **START**, change pose, then save **TARGET**. Viewers can compare Hi3D mesh, PBR texture, TokenRig and SOMA, and inspect left/right anatomy before generating motion.

Once both poses are saved, the animation cells use `viewer_poses.json` automatically. Blender is invoked **headlessly as a converter/exporter**, with no Blender GUI and no need to leave Colab.


In [ ]:
# Interactive Colab 3D studio: starts a local viewer server, no external web app.
from google.colab import output
import sys, socket
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from scripts.viewer_server import launch as launch_i3d_viewer
PORT = 8765
if "_i3d_viewer" not in globals():
    _i3d_viewer = launch_i3d_viewer(WORK, port=PORT)
    print("I-3D viewer server started", PORT)
else:
    print("Reusing existing I-3D viewer server", PORT)
output.serve_kernel_port_as_iframe(PORT, height=950)


## 11. Optional numeric fallback if you don't use the 3D Studio pose editor

The interactive viewer is the primary way to pose the real 3D character. The values below are **fallback poses only**, used if both START and TARGET have **not** been saved in the viewer. Editing these values is optional. If you saved the two poses in the viewer, they take precedence.


In [ ]:
# Customize the target pose of the actual 3D rig.
TARGET_ROOT_OFFSET_METERS = [0.6, 0.0, 0.0]   # Blender XYZ, Z-up; move 0.6m right
TARGET_BONE_ROTATIONS_DEGREES = {
    "LeftArm": [0.0, 0.0, 20.0],
    "RightArm": [0.0, 0.0, -20.0],
    "LeftForeArm": [0.0, 0.0, -25.0],
    "RightForeArm": [0.0, 0.0, 25.0],
}
# Remove a key or set all angles to zero if the pose deformation looks wrong.
POSE_BLEND = WORK / "character_poses.blend"
print("Target displacement:", TARGET_ROOT_OFFSET_METERS)
print("Joint changes:", TARGET_BONE_ROTATIONS_DEGREES)


In [ ]:
# Write a Blender-only authoring script. It edits the skinned SOMA model directly.
POSE_SCRIPT = WORK / "author_poses_colab.py"
POSE_SCRIPT.write_text(r'''
import bpy, json, math, sys
from pathlib import Path
from mathutils import Vector
p = json.loads(Path(sys.argv[sys.argv.index("--")+1]).read_text())
bpy.ops.object.select_all(action="SELECT")
bpy.ops.object.delete(use_global=False)
bpy.ops.import_scene.gltf(filepath=p["character"])
arms = [a for a in bpy.context.scene.objects if a.type=="ARMATURE" and a.name.startswith("I3D_SOMA")]
if len(arms)!=1: raise ValueError("Require one I3D_SOMA armature")
arm=arms[0]
if len(arm.pose.bones)!=30: raise ValueError("Expected 30 SOMA bones")
if not all(abs(float(arm.matrix_world[i][j]) - (1.0 if i==j else 0.0))<1e-4 for i in range(4) for j in range(4)):
    raise ValueError("Armature global transform must be identity")
scene=bpy.context.scene
scene.frame_set(1)
for pb in arm.pose.bones:
    pb.rotation_mode="XYZ"
    pb.rotation_euler=(0.0,0.0,0.0)
    pb.location=(0.0,0.0,0.0)
    pb.scale=(1.0,1.0,1.0)
    for field in ("location","rotation_euler","scale"):
        pb.keyframe_insert(data_path=field,frame=1)
scene.frame_set(2)
for name,xyz in p["angles"].items():
    if name not in arm.pose.bones: raise ValueError("Unknown SOMA bone: "+name)
    arm.pose.bones[name].rotation_mode="XYZ"
    arm.pose.bones[name].rotation_euler=[math.radians(v) for v in xyz]
bpy.context.view_layer.update()
hips=arm.pose.bones["Hips"]
matrix=hips.matrix.copy()
matrix.translation+=Vector(p["offset"])
hips.matrix=matrix
bpy.context.view_layer.update()
for pb in arm.pose.bones:
    for field in ("location","rotation_euler","scale"):
        pb.keyframe_insert(data_path=field,frame=2)
scene.frame_set(1)
scene.render.fps=30
bpy.ops.wm.save_as_mainfile(filepath=p["blend"])
print("I3D_POSES_SAVED",p["blend"])
''', encoding="utf-8")
POSE_CONFIG = WORK / "pose_config.json"
POSE_CONFIG.write_text(json.dumps({
    "character":str(SOMA_GLB), "blend":str(POSE_BLEND),
    "angles":TARGET_BONE_ROTATIONS_DEGREES,
    "offset":TARGET_ROOT_OFFSET_METERS
}, indent=2))
# The viewer-authored poses take priority; the numerical script above is fallback only.
POSES_FROM_VIEWER = WORK / "viewer_poses.json"
USE_VIEWER_POSES = False
if POSES_FROM_VIEWER.is_file():
    _viewer = json.loads(POSES_FROM_VIEWER.read_text(encoding="utf-8"))
    USE_VIEWER_POSES = {"start","target"}.issubset(_viewer.get("poses", {}))
if USE_VIEWER_POSES:
    print("Using START and TARGET edited on the actual 3D character inside Colab.")
    run(BLENDER, "-b", "--python", ROOT/"scripts"/"author_viewer_poses.py", "--",
        "--character", SOMA_GLB, "--poses", POSES_FROM_VIEWER, "--output", POSE_BLEND)
else:
    print("No complete viewer pose pair found; using numeric sample pose settings.")
    run(BLENDER, "-b", "--python", POSE_SCRIPT, "--", POSE_CONFIG)
required(POSE_BLEND)
print("Two 3D poses ready:", POSE_BLEND)



### Optional: replace the generated pose file with a manually edited Blender file

Run this cell *only if* you edited the exported SOMA mesh and its two keyframes on another machine. Otherwise keep the `.blend` file generated above.


In [ ]:
# OPTIONAL — uncomment these lines to upload an edited .blend instead:
# from google.colab import files
# incoming = files.upload()
# blendname, blendbytes = next(iter(incoming.items()))
# if not blendname.lower().endswith(".blend"):
#     raise ValueError("Expected a Blender .blend file")
# POSE_BLEND.write_bytes(blendbytes)
# print("Using manually posed 3D rig:", POSE_BLEND)


## 12. Generate MotionBricks animation and export the same textured character

The original I-3D scripts export frame 1 as **four repeated starting context poses** (stationary start) and frame 2 as **one target pose** (arrival/stop). MotionBricks creates the intermediate 30 FPS skeleton animation, after which Blender applies those transformations to the **same** skinned character and exports the animated GLB plus editable `.blend`.

This is **one short transition** (roughly 0.6–1.9 seconds). Continuous sequences require further target poses and chained generations.


In [ ]:
POSES_JSON = WORK / "05_poses.json"
MOTION_NPZ = WORK / "06_motion.npz"
ANIMATED_GLB = WORK / "07_animated.glb"

run(BLENDER, POSE_BLEND, "-b", "--python", ROOT/"scripts"/"export_two_poses.py",
    "--", "--output", POSES_JSON)
required(POSES_JSON)

run(MOTION_PY, ROOT/"scripts"/"generate_motion.py",
    "--poses", POSES_JSON, "--reference", REFERENCE, "--bridge", BRIDGE,
    "--output", MOTION_NPZ, "--precision", MOTION_PRECISION)
required(MOTION_NPZ)

run(BLENDER, "-b", "--python", ROOT/"scripts"/"apply_motion_blender.py", "--",
    "--character", SOMA_GLB, "--reference", REFERENCE, "--bridge", BRIDGE,
    "--motion", MOTION_NPZ, "--output", ANIMATED_GLB)
required(ANIMATED_GLB)
print("MotionBricks animated and textured character:", ANIMATED_GLB)


## Watch the final MotionBricks animation inside Colab

The same I-3D viewer now loads `07_animated.glb`. Click **Refresh outputs**, select **MotionBricks animation**, then Play/Pause, scrub or record WebM video. You can compare it side-by-side with the earlier meshes and change materials, lighting, skeleton and render diagnostics. No external application or file download is needed to watch the result.


In [ ]:
# Interactive Colab 3D studio: starts a local viewer server, no external web app.
from google.colab import output
import sys, socket
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from scripts.viewer_server import launch as launch_i3d_viewer
PORT = 8765
if "_i3d_viewer" not in globals():
    _i3d_viewer = launch_i3d_viewer(WORK, port=PORT)
    print("I-3D viewer server started", PORT)
else:
    print("Reusing existing I-3D viewer server", PORT)
output.serve_kernel_port_as_iframe(PORT, height=950)


## 13. Download final outputs

Download the final animated GLB, the editable Blender scene, the prepared skinned SOMA character, the pose data, and the bridge diagnostics. The actual results are only generated after **you run** the inference cells on a compatible GPU.

To preview the GLB, use Blender's Material Preview/Rendered viewport, a glTF animation viewer, or import it into an engine with an animated glTF loader.


In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED
DELIVERY = WORK / "I3D_generated_character_and_animation.zip"
artifacts = [IMAGE, GEOMETRY, TEXTURED, TOKENRIG_GLB, SOMA_GLB, BRIDGE, REFERENCE,
             POSE_BLEND, POSES_JSON, MOTION_NPZ, ANIMATED_GLB,
             ANIMATED_GLB.with_suffix(".blend")]
with ZipFile(DELIVERY, "w", ZIP_DEFLATED) as bundle:
    for path in artifacts:
        if Path(path).is_file():
            bundle.write(path, Path(path).name)
print("Output bundle:", DELIVERY)
files.download(str(DELIVERY))


## Source links and attribution

- [I-3D project source, README and original bridge](https://github.com/Apache0ne/I-3D)
- [Hi3D v1.0](https://github.com/hi3d-official/Hi3Dv1.0) / [weights](https://huggingface.co/hi3d-official/Hi3Dv1.0)
- [Hunyuan3D-2mini](https://github.com/Tencent-Hunyuan/Hunyuan3D-2) / [weights](https://huggingface.co/tencent/Hunyuan3D-2mini)
- [Hunyuan3D-Paint 2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1)
- [SkinTokens / TokenRig](https://github.com/VAST-AI-Research/SkinTokens)
- [NVIDIA SOMA-X](https://github.com/NVlabs/SOMA-X)
- [NVIDIA MotionBricks](https://github.com/NVlabs/motionbricks)

See the original LICENSE and NOTICE files under `vendor/`, and the I-3D README for separate code, model-weight and sample-motion restrictions. The approximate spatial SOMA skin bridge is original I-3D code rather than a certified NVIDIA converter. **Notebook was written without running or validating the GPU stages.**
